In [326]:
import torch
import numpy as np
import random
from torch.utils.data import DataLoader
import os
import urllib
import zipfile
import lxml.etree
import re
from collections import Counter

In [327]:
if not os.path.isfile('ted_en-20160408.xml'):
    urllib.request.urlretrieve("https://github.com/oxford-cs-deepnlp-2017/practical-1/blob/master/ted_en-20160408.xml?raw=true", filename="ted_en-20160408.xml")

In [328]:
doc = lxml.etree.parse('ted_en-20160408.xml')
input_text = doc.xpath('//content/text()')
label = doc.xpath('//head/keywords/text()')
del doc
len(input_text)

2085

In [329]:
# Preprocess sentences to exclude all characters except alphabets and numbers
texts = [re.sub(r'\([^)]*\)', '',text) for text in input_text]
texts = [re.sub('r([^a-zA-Z0-9\s])',' ',text) for text in texts] #Included '.'
texts = [re.sub('[^a-zA-Z0-9\']',' ',text) for text in texts] #To replace '.' with ' '
texts = [re.sub('[^a-zA-Z0-9 ]','',text) for text in texts]
texts = [text.lower() for text in texts] #uppercase->lowercase

In [330]:
texts_labels=zip(texts,label)
texts = [text_label for text_label in texts_labels if len(text_label[0]) > 500]
print('number of text greater than 500 words are:',len(texts))

number of text greater than 500 words are: 2076


In [331]:
texts,labels=zip(*texts)
print(labels[:10])

('talks, business, creativity, curiosity, goal-setting, innovation, motivation, potential, success, work', 'talks, Planets, TEDx, bacteria, biology, engineering, environment, evolution, exploration, future, innovation, intelligence, microbiology, nature, potential, science', 'talks, Debate, Guns, activism, big problems, children, choice, community, future, goal-setting, government, law, leadership, marketing, parenting, policy, social change, violence', 'talks, Brazil, Slavery, art, beauty, community, creativity, culture, design, global issues, humanity, identity, photography, race, social change, society, visualizations', 'talks, NASA, communication, computers, creativity, design, engineering, exploration, future, innovation, interface design, invention, microsoft, potential, prediction, product design, technology, visualizations', 'talks, Africa, Internet, community, democracy, development, future, government, identity, leadership, politics, potential', 'talks, ancient world, animals

In [332]:
tokens = [words for text in texts for words in text.split()]
words_count = Counter(tokens)
words_least_common = set([word for word,count in words_count.most_common() if count==1])

In [1]:
texts = [[word for word in text.split() if word not in words_least_common]for text in texts]

NameError: name 'texts' is not defined

In [334]:
all_labels=[]
for i,keyword in enumerate(labels):
    key = keyword.split(', ')
    all_labels+=key

all_labels_set = set(all_labels)

In [335]:
label2id = {label: idx for idx, label in enumerate(all_labels_set)}
id2label = {idx: label for label, idx in label2id.items()}

In [336]:
count_labels=Counter(all_labels)
label_count = [word_count for word_count in count_labels.most_common()]
label_count

[('talks', 2076),
 ('TED Conference', 698),
 ('technology', 595),
 ('culture', 451),
 ('science', 447),
 ('global issues', 428),
 ('design', 352),
 ('TEDx', 321),
 ('business', 289),
 ('entertainment', 262),
 ('arts', 183),
 ('education', 159),
 ('politics', 157),
 ('health', 155),
 ('creativity', 147),
 ('art', 135),
 ('economics', 119),
 ('medicine', 118),
 ('biology', 117),
 ('TED Fellows', 114),
 ('brain', 109),
 ('music', 102),
 ('cities', 101),
 ('social change', 100),
 ('invention', 99),
 ('storytelling', 97),
 ('environment', 96),
 ('activism', 89),
 ('children', 87),
 ('health care', 87),
 ('innovation', 86),
 ('future', 83),
 ('women', 83),
 ('war', 83),
 ('history', 82),
 ('psychology', 81),
 ('photography', 80),
 ('animals', 80),
 ('collaboration', 76),
 ('humor', 76),
 ('communication', 73),
 ('Africa', 71),
 ('computers', 69),
 ('architecture', 66),
 ('exploration', 63),
 ('society', 63),
 ('oceans', 60),
 ('nature', 59),
 ('performance', 59),
 ('happiness', 57),
 ('physi

In [337]:
labels_indices = []
for keyword in labels:
    first_label = keyword.split(', ')[1]  # Take only the first label
    labels_indices.append(label2id[first_label])
# Check the labels indices for the first few samples
print(labels_indices[:10])  # This will print the indices of the labels for the first 10 samples

[266, 34, 30, 260, 42, 218, 5, 340, 118, 348]


In [338]:
tokens.append('<UNK>')
tokens.append('<PAD>')

In [339]:
vocab = list(set(tokens))

In [340]:
print('size of vocabulary:',len(vocab))
id2word = dict(enumerate(vocab))
word2id = dict((val,key) for (key,val) in id2word.items())

size of vocabulary: 57042


In [341]:
# Stripping Text to fall within length of 500; incase if it is shorter then padd with '<UNK>'
length = 500 #sentence length
stripped_text = []#np.zeros((len(texts),length)
for i,text in enumerate(texts):
    inputs = []
    if len(text) >= 500:
        inputs.extend(text[:500])
    else:
        extra_length = 500-len(text)
        extra = ['<PAD>']*extra_length
        word_with_extra = text + extra
        inputs.extend(word_with_extra)
    stripped_text.append(inputs) 

In [342]:
stripped_length = len(stripped_text)
print(stripped_length)

2076


In [343]:
inputs = []
text_ids = []
for text in stripped_text:
    for word in text:
        i = word2id[word]
        inputs.append(i)
    text_ids.append(inputs)
    inputs = []

In [ ]:
data = list(zip(text_ids, labels_indices))  # Each item is a tuple (text, [label_indices])

In [345]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (80-20 split)
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Separate the inputs (text) and labels for train and test data
train_texts, train_labels = zip(*train_data)
test_texts, test_labels = zip(*test_data)

# ✅ Convert everything to tensors for PyTorch (single label per example)
train_texts_tensor = torch.tensor(train_texts, dtype=torch.long)
train_labels_tensor = torch.tensor(train_labels, dtype=torch.long)  # <— FIXED

test_texts_tensor = torch.tensor(test_texts, dtype=torch.long)
test_labels_tensor = torch.tensor(test_labels, dtype=torch.long)  

In [346]:
class TextGenerationModel(torch.nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, num_layers=3):
        super(TextGenerationModel, self).__init__()
        
        # Embeddings for input text and labels
        self.embedding = torch.nn.Embedding(vocab_size, embedding_dim)
        self.label_embedding = torch.nn.Embedding(num_labels, embedding_dim)  # Embedding for label indices
        self.dropout = torch.nn.Dropout(0.5)
        self.linear1 = torch.nn.Linear(embedding_dim*2,embedding_dim*2)
        self.dropout2= torch.nn.Dropout(0.25)
        
        # LSTM for sequence processing
        self.rnn = torch.nn.LSTM(embedding_dim*2, hidden_dim, num_layers, batch_first=True)
        self.fc_out = torch.nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, label):
        # Embed the text and label
        embedded_text = self.embedding(x)
        label_emb = self.label_embedding(label).unsqueeze(1).repeat(1, embedded_text.size(1), 1)
        
        # Concatenate text and label embeddings
        combined = torch.cat((embedded_text, label_emb), dim=-1)
        combined = self.dropout(combined)

        combined=self.linear1(combined)
        combined=self.dropout2(combined)
        # Pass through LSTM
        rnn_out, _ = self.rnn(combined)
        
        # Output layer to predict next token in the sequence
        output = self.fc_out(rnn_out)
        
        return output


In [360]:
import torch.nn as nn

class TextGenerationModel2(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_labels, dropout=0.3, nhead=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.label_embedding = nn.Embedding(num_labels, embedding_dim)
        self.dropout = nn.Dropout(dropout)

        self.positional_encoding = nn.Parameter(torch.randn(1, 512, embedding_dim * 2))  # Max length 512

        self.transformer_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim * 2,
            nhead=nhead,
            dim_feedforward=hidden_dim,
            dropout=dropout,
            batch_first=True
        )
        self.fc = nn.Linear(embedding_dim * 2, vocab_size)

    def forward(self, text, label):
        embedded_text = self.embedding(text)
        label_emb = self.label_embedding(label).unsqueeze(1).repeat(1, embedded_text.size(1), 1)
        
        x = torch.cat((embedded_text, label_emb), dim=-1)
        x = x + self.positional_encoding[:, :x.size(1), :]  # Add positional encoding
        x = self.dropout(x)
        
        transformer_out = self.transformer_layer(x)  # [batch, seq_len, d_model]
        output = self.fc(transformer_out)
        return output


In [362]:
import torch.optim as optim

# Hyperparameters
embedding_dim = 128
hidden_dim = 256
num_labels = len(all_labels_set)  # Number of unique labels
vocab_size = len(vocab)
num_epochs = 1

# Initialize model and optimizer
model = TextGenerationModel2(vocab_size, embedding_dim, hidden_dim, num_labels)
optimizer = optim.Adam(model.parameters(), lr=0.001)
print(f"Model initialized with vocab_size={vocab_size}, embedding_dim={embedding_dim}, hidden_dim={hidden_dim}, num_labels={num_labels}")
print(f"Starting training for {num_epochs} epochs...\n")

# Training loop
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")
    for i in range(len(train_texts_tensor)):
        optimizer.zero_grad()
        
        text_input = train_texts_tensor[i].unsqueeze(0)  # Add batch dimension
        label_input = train_labels_tensor[i].unsqueeze(0) 
        print(f"\nSample {i + 1}/{len(train_texts_tensor)}")
        print(f"  text_input shape: {text_input.shape}")
        print(f"  label_input shape: {label_input.shape}")
        output = model(text_input, label_input)  # Get model output
        print(f"  model output shape: {output.shape}")
        
        # Calculate loss (cross-entropy)
        loss = torch.nn.CrossEntropyLoss()(output.view(-1, vocab_size), text_input.view(-1))  # Next word prediction
        print(f"  loss: {loss.item():.4f}")
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_texts_tensor)
    print(f"\n✅ Epoch {epoch + 1} complete. Average Loss: {avg_loss:.4f}")

# Save the trained model
torch.save(model.state_dict(), 'text_generation_model2.pth')


Model initialized with vocab_size=57042, embedding_dim=128, hidden_dim=256, num_labels=367
Starting training for 1 epochs...


--- Epoch 1/1 ---

Sample 1/1660
  text_input shape: torch.Size([1, 500])
  label_input shape: torch.Size([1])
  model output shape: torch.Size([1, 500, 57042])
  loss: 11.1118

Sample 2/1660
  text_input shape: torch.Size([1, 500])
  label_input shape: torch.Size([1])
  model output shape: torch.Size([1, 500, 57042])
  loss: 11.0238

Sample 3/1660
  text_input shape: torch.Size([1, 500])
  label_input shape: torch.Size([1])
  model output shape: torch.Size([1, 500, 57042])
  loss: 10.7836

Sample 4/1660
  text_input shape: torch.Size([1, 500])
  label_input shape: torch.Size([1])
  model output shape: torch.Size([1, 500, 57042])
  loss: 10.9160

Sample 5/1660
  text_input shape: torch.Size([1, 500])
  label_input shape: torch.Size([1])
  model output shape: torch.Size([1, 500, 57042])
  loss: 10.8596

Sample 6/1660
  text_input shape: torch.Size([1, 500])
  la

In [ ]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")
    
    for i in range(len(train_texts_tensor)):
        optimizer.zero_grad()

        full_text = train_texts_tensor[i]  # shape: [seq_len]
        label_input = train_labels_tensor[i].unsqueeze(0)  # shape: [1]

        if full_text.size(0) < 2:
            continue  # skip short sequences

        input_seq = full_text[:-1].unsqueeze(0)   # [1, seq_len - 1]
        target_seq = full_text[1:].unsqueeze(0)   # [1, seq_len - 1]

        print(f"\nSample {i + 1}/{len(train_texts_tensor)}")
        print(f"  input_seq shape: {input_seq.shape}")
        print(f"  target_seq shape: {target_seq.shape}")
        print(f"  label_input shape: {label_input.shape}")
        
        output = model(input_seq, label_input)  # output shape: [1, seq_len - 1, vocab_size]
        print(f"  model output shape: {output.shape}")

        loss = torch.nn.CrossEntropyLoss()(output.view(-1, vocab_size), target_seq.view(-1))
        print(f"  loss: {loss.item():.4f}")

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_texts_tensor)
    print(f"\n✅ Epoch {epoch + 1} complete. Average Loss: {avg_loss:.4f}")


In [382]:

def generate_ted_talk(model, seed_text, label_name, vocab, word2id, id2word, all_labels_set, max_length=100):
    model.eval()

    # Map label to index
    label_to_idx = {label: i for i, label in enumerate(all_labels_set)}
    label_idx = label_to_idx[label_name]
    label_tensor = torch.tensor([label_idx], dtype=torch.long)

    # Tokenize seed text
    seed_tokens = seed_text.lower().strip().split()
    print(seed_tokens)
    seed_token_ids = [word2id.get(token, word2id.get('<unk>', 0)) for token in seed_tokens]
    print(seed_token_ids)
    generated_ids = seed_token_ids.copy()

    for _ in range(max_length):
        input_tensor = torch.tensor(generated_ids, dtype=torch.long)
        
        # Ensure input_tensor is [1, seq_len]
        if input_tensor.dim() == 1:
            input_tensor = input_tensor.unsqueeze(0)

        with torch.no_grad():
            output = model(input_tensor, label_tensor)  # shape: [1, seq_len, vocab_size]
            next_token_logits = output[0, -1]  # get logits for last time step
            temperature = 2.0  # You can set this to 0.7, 1.0, 1.5, etc.
            probs = torch.softmax(next_token_logits / temperature, dim=0)
            next_token_id = torch.multinomial(probs, num_samples=1).item()
            generated_ids.append(next_token_id)

            # Optional: stop if end-of-sequence token generated
            if id2word[next_token_id] == '<eos>':
                break

    # Convert token ids back to words
    generated_words = [id2word.get(idx, '<UNK>') for idx in generated_ids]
    return ' '.join(generated_words)


In [370]:
def generate_ted_talk_with_repetition_penalty(model, seed_text, label_name, word2id, id2word, label2id, max_len=100, temperature=1.0, repetition_penalty=1.2):
    model.eval()

    input_ids = [word2id.get(w, word2id["<UNK>"]) for w in seed_text.split()]
    input_tensor = torch.tensor(input_ids).unsqueeze(0)
    label_tensor = torch.tensor([label2id[label_name]])

    generated_ids = input_ids.copy()

    for _ in range(max_len):
        with torch.no_grad():
            context_window = 32  # or whatever your model was trained with
            current_input = torch.tensor(generated_ids[-context_window:]).unsqueeze(0)
            current_input = torch.tensor(generated_ids).unsqueeze(0)
            output = model(current_input, label_tensor)  # [1, seq_len, vocab_size]
            logits = output[0, -1, :]  # last token logits: [vocab_size]

            # 🔁 Apply repetition penalty
            for token_id in set(generated_ids):
                logits[token_id] /= repetition_penalty

            # 🔥 Apply temperature
            probs = torch.softmax(logits / temperature, dim=0)

            # 🎲 Sample
            next_token_id = torch.multinomial(probs, num_samples=1).item()

            generated_ids.append(next_token_id)

            # Optional: break on EOS token
            if id2word[next_token_id] == "<eos>":
                break

    return " ".join(id2word[i] for i in generated_ids[len(input_ids):])


In [350]:
print(word2id.get("compensating"))
print(word2id.get("I"))

0
None


In [383]:
seed_text = "this"
label_name = "technology"  # Must exist in all_labels_set
# Hyperparameters
embedding_dim = 128
hidden_dim = 256
num_labels = len(all_labels_set)  # Number of unique labels
vocab_size = len(vocab)
num_epochs = 10

# Initialize model and optimizer
model = TextGenerationModel2(vocab_size, embedding_dim, hidden_dim, num_labels)
model.load_state_dict(torch.load('text_generation_model2.pth'))
model.eval()

generated_talk = generate_ted_talk(
    model,
    seed_text=seed_text,
    label_name=label_name,
    vocab=vocab,
    word2id=word2id,
    id2word=id2word,
    all_labels_set=all_labels_set,
    max_length=100
)

print("\nGenerated TED Talk:\n")
print(generated_talk)


['this']
[15171]

Generated TED Talk:

this socialness do do do do do do coils gi philippines stumbles dele 1877 transport cmere corporation 2070 rivulet vitra clerical extraneous blended bulgarians zafa lowercased gazuntas yukon heaters eardrum car angelfish entrust aground legislate neurological middles insulin sprouts brush configuration crocodiles heavens workaround admittedly righthand represents roadrunner faking harmony practitione trendiest flatte chokes lend childlike skipped rudder vagrants fading six bark microfiltration gibsons kiddies grisly pursue friendliness algal incorporates defiling fermentation carbonex indian bailey ashram 433 depressive luscious umbrellas fulton sterling banduras meeting meeting outmatch comma touati caw kidman sectional saujani rationalist flirting germ jab diclofenac smog rescinded taiwan zappe


In [384]:
seed_text = "<BOS>"
label_name = "parenting"  # Must exist in all_labels_set
# Hyperparameters
embedding_dim = 128
hidden_dim = 256
num_labels = len(all_labels_set)  # Number of unique labels
vocab_size = len(vocab)
num_epochs = 10

# Initialize model and optimizer
model = TextGenerationModel2(vocab_size, embedding_dim, hidden_dim, num_labels)
model.load_state_dict(torch.load('text_generation_model2.pth'))
model.eval()

seed_text = "this"
label_name = "parenting"  # must be in label2id

generated = generate_ted_talk_with_repetition_penalty(
    model=model,
    seed_text=seed_text,
    label_name=label_name,
    word2id=word2id,
    id2word=id2word,
    label2id=label2id,
    max_len=100,
    temperature=0.5,
    repetition_penalty=3
)

print("\nGenerated TED Talk:\n")
print(generated)



Generated TED Talk:

arrived service massive definition aggressive historical white bridges collect do relatives chimpanzee blind depressed take collaborative youre painful content valley behavior persons diving 500 breast measuring slaves mere infection furthe random beat 4 shake dependent stark fact private regime skeletons direction destroyed wheels doubt hulk trail know drove grown picking crew founding battle was knee pages photographe medications permanently entrepreneurs eating dust muslim tables some preventable way thus sheer and territory division beaches jean quietly acto frustrated entirely already marry 160 1990s need marketing practices column large minded nobodys iconic shadow modern reluctant windows tricky be profoundly chapter shop numbe
